In [0]:
import sys
import pyspark.sql.functions as F
#from datetime import datetime, timedelta

class BatchBalance:
  
  # The constuctor takes 2 arguments, 
    # 1. loggerObj - this should have methods info, debug and exception with all taking string messages as arguments for logging.
    # 2. abcDbObj - this should have methods executeSelectQuery(return result as dataframe) & executeDmlQuery which takes string SQL quries as arguments.
  def __init__(self, loggerObj, abcDbObj):
    self.logger = loggerObj
    self.moduleName = "BATCH_BALANCE"
    self.abcDbObj = abcDbObj
    self.logger.info(f"{self.moduleName} | BatchBalance class initialized")
    pass
 
  #------------------------------------------------------------------------------------------------------------------------------------#
  #===================================================GET BATCH SET IN PROGRESS========================================================#
  #------------------------------------------------------------------------------------------------------------------------------------#
  def getBatchSetInProgress(self, batchSetCategory, batchSetSrcSysNme):
    try:
      self.logger.info(f"{self.moduleName} | -----getBatchSetInProgress method START-------")
      self.logger.debug(f"{self.moduleName} | Getting the batch set in progress with batch_set_category - {batchSetCategory} and batch_set_src_sys_nme - {batchSetSrcSysNme}")
      batchSetFetchQuery = f"select * from batch.t_batch_set  where batch_set_category = '{batchSetCategory}' and batch_set_src_sys_nme = '{batchSetSrcSysNme}' and batch_set_sts_ind = 'P'"
      self.logger.debug(f"{self.moduleName} | batch_set_fetch_query - {batchSetFetchQuery}")
      batchSetDetails = self.abcDbObj.executeSelectQuery(batchSetFetchQuery)
      self.logger.debug("-----getBatchSetInProgress method END---------")
      return batchSetDetails
    except Exception:
      self.logger.exception(f"{self.moduleName} | Exception while retreiving in progress batchset details for batchSetCategory:{batchSetCategory}, batchSetSrcSysNme:{batchSetSrcSysNme} . ")
      raise Exception
  #------------------------------------------------------------------------------------------------------------------------------------#
  #===================================================GET BATCH IN PROGRESS============================================================#
  #------------------------------------------------------------------------------------------------------------------------------------#    
  def getBatchInProgress(self, batchSetCategory, batchSetSrcSysNme, batchNme):
    
    try:
      self.logger.info(f"{self.moduleName} | -----getBatchInProgress method START-------")
      self.logger.debug(f"{self.moduleName} | Getting the batch in progress with batch_set_category - {batchSetCategory}, batch_nme - {batchNme} and  batch_set_src_sys_nme - {batchSetSrcSysNme}")
      fetchBatchQuery = f"SELECT t_batch.batch_key,t_batch.batch_nme,t_batch_set.batch_set_category,t_batch.batch_sts_ind,t_batch_set.batch_set_src_sys_nme  FROM  batch.t_batch t_batch join batch.t_batch_set  t_batch_set  on  t_batch.batch_set_key=t_batch_set.batch_set_key  where batch_sts_ind='P' and batch_set_category = '{batchSetCategory}' and batch_nme = '{batchNme}' and batch_set_src_sys_nme = '{batchSetSrcSysNme}'"
      self.logger.debug(f"{self.moduleName} | fetchBatchQuery - {fetchBatchQuery}")
      batchDetails = self.abcDbObj.executeSelectQuery(fetchBatchQuery)
      self.logger.info(f"{self.moduleName} | -----getBatchInProgress method END---------")
      return batchDetails
    
    except Exception:
      self.logger.exception(f"{self.moduleName} | Exception while retreiving in progress batch details for batchSetCategory:{batchSetCategory}, batchSetSrcSysNme:{batchSetSrcSysNme}, batchNme:{batchNme} .")
      raise Exception
      
  #------------------------------------------------------------------------------------------------------------------------------------#
  #=========================================================OPEN BATCH SET=============================================================#
  #------------------------------------------------------------------------------------------------------------------------------------#
  def openBatchSet(self, batchSetCategory, batchSetSrcSysNme, batchSetSrcSysCd):
    try:
      self.logger.info(f"{self.moduleName} | -----openBatchSet function START-------")
      batch_set_key = -1#
      self.logger.debug(f"{self.moduleName} | Creating batch set entry for : batch_set_category-{batchSetCategory}, batch_set_src_sys_nme-{batchSetSrcSysNme}, batch_set_src_sys_cd - {batchSetSrcSysCd}")
      progressingBatchSetDf = self.getBatchSetInProgress(batchSetCategory, batchSetSrcSysNme)
      progressingBatchSet = progressingBatchSetDf.collect()[0] if progressingBatchSetDf.count()>0 else None
      

      if (progressingBatchSet != None):
        raise Exception(f"{self.moduleName} | Same Batch set in progress with batch set key as - {progressingBatchSet['batch_set_key']}.. it has to be completed before opening the new one !!!")
      else:
        params_batch_set_insert =(batchSetCategory,batchSetSrcSysNme,batchSetSrcSysCd)
        insertQuery = f"""INSERT INTO [batch].[t_batch_set] ([batch_set_category], [batch_set_src_sys_nme], [batch_set_src_sys_cd], [batch_set_cfg_value], [batch_set_strt_tm], [batch_set_end_tm], [batch_set_sts_ind], [batch_set_sts_descr], [batch_force_run_ind]) VALUES (?, ?, ?, 'F', SYSDATETIME(), NULL, 'P', 'Progress', 'N')"""
        self.logger.debug(f"{self.moduleName} | insertQuery - {insertQuery}")
        self.abcDbObj.executeDmlQuery(insertQuery, params_batch_set_insert)
        batchSetKey = self.getBatchSetInProgress(batchSetCategory, batchSetSrcSysNme).collect()[0]['batch_set_key']
      self.logger.info(f"{self.moduleName} | -----openBatchSet function END---------")
      return batchSetKey
    except Exception:
      self.logger.exception(f"{self.moduleName} | Exception while opening batchset for batchSetCategory:{batchSetCategory}, batchSetSrcSysNme:{batchSetSrcSysNme}, batchSetSrcSysCd:{batchSetSrcSysCd}")
      raise Exception

  #------------------------------------------------------------------------------------------------------------------------------------#
  #=========================================================CLOSE BATCH SET============================================================#
  #------------------------------------------------------------------------------------------------------------------------------------#    
  def closeBatchSet(self, batchSetCategory, batchSetSrcSysNme, batchSetSrcSysCd, batchLoadDifference = 0):
    try:
      self.logger.info(f"{self.moduleName} | -----closeBatchSet function START-------")
      batchSetInProgDf = self.getBatchSetInProgress(batchSetCategory, batchSetSrcSysNme)
      if batchSetInProgDf.count() == 0:
        raise Exception(f"No batchset in progress for batchSetCategory: {batchSetCategory}, batchSetSrcSysNme:{batchSetSrcSysNme}, batchSetSrcSysCd:{batchSetSrcSysCd} to be closed")
      
      batchSetKey = batchSetInProgDf.collect()[0]['batch_set_key']
      emptyBatchSetCheckQuery = f"SELECT batch_key  FROM  batch.t_batch where batch_set_key= {batchSetKey}"
      self.logger.debug(f"{self.moduleName} | emptyBatchSetCheckQuery - {emptyBatchSetCheckQuery}")
      batchCount = self.abcDbObj.executeSelectQuery(emptyBatchSetCheckQuery).count()
      fetchBatchStatusQuery = f"SELECT t_batch.batch_key  FROM  batch.t_batch where batch_set_key= {batchSetKey} and batch_sts_ind !='S'"
      self.logger.debug(f"{self.moduleName} | fetch_batch_query - {fetchBatchStatusQuery}")
      batchStatusDetailsDf = self.abcDbObj.executeSelectQuery(fetchBatchStatusQuery)
      batchStatusDetails = batchStatusDetailsDf.collect()[0]['batch_key'] if batchStatusDetailsDf.count() > 0  else None
      self.logger.debug(f"{self.moduleName} | Closing the batch set entry -> batch_set_category:{batchSetCategory}, batch_set_src_sys_nme:{batchSetSrcSysNme}, batch_set_src_sys_cd:{batchSetSrcSysCd}")
      if (batchCount == 0 or batchStatusDetails != None):
        batchSetStsInd = 'F' 
        batchSetStsDescr = 'Failure' 
      else:
        batchSetStsInd = 'S' 
        batchSetStsDescr = 'Success' 
      
      params_close_batch_set=(batchSetStsInd,batchSetStsDescr,batchSetKey)  
      updateQuery = f"update [batch].[t_batch_set] set batch_set_sts_ind = ?, batch_set_sts_descr = ?, batch_set_end_tm = SYSDATETIME() where batch_set_key = ?"
      self.logger.debug(f"{self.moduleName} | updateQuery - {updateQuery}")
      self.abcDbObj.executeDmlQuery(updateQuery,params_close_batch_set)
      return batchSetStsInd
      self.logger.info(f"{self.moduleName} | -----closeBatchSet function END---------")
    except Exception:
      self.logger.exception(f"{self.moduleName} | Exception while closing batchset for batchSetCategory:{batchSetCategory}, batchSetSrcSysNme:{batchSetSrcSysNme}, batchSetSrcSysCd:{batchSetSrcSysCd}")
      raise Exception

      
  #------------------------------------------------------------------------------------------------------------------------------------#
  #======================================================UPDATE BATCH SET STATUS=======================================================#
  #------------------------------------------------------------------------------------------------------------------------------------# 
  def updateBatchSetStatus(self,batchSetCategory, batchSetSrcSysNme, batchSetSrcSysCd, batchSetStatusInd):
    try:
      self.logger.info(f"{self.moduleName} | -----updateBatchSetStatus function START-------")
      batchSetKey = None
      batchSetStatusDict = {'F':'Failure','S':'Success','P':'Progress'}
      batchSetStatusInd = batchSetStatusInd.strip().upper()
      
      batchSetInProgDf = self.getBatchSetInProgress(batchSetCategory, batchSetSrcSysNme)
      if batchSetInProgDf.count() == 0:
        raise Exception(f"No batchset in progress for batchSetCategory: {batchSetCategory}, batchSetSrcSysNme:{batchSetSrcSysNme}, batchSetSrcSysCd:{batchSetSrcSysCd} to be updated")

      batchSetKey = batchSetInProgDf.collect()[0]['batch_set_key']
      params_update_batch_set = (batchSetStatusInd,batchSetStatusDict[batchSetStatusInd],batchSetKey)
      updateQuery = f"UPDATE batch.t_batch_set SET batch_set_sts_ind = ?,batch_set_sts_descr = ?,batch_set_end_tm = SYSDATETIME() where batch_set_key = ? "
      self.logger.debug(f"{self.moduleName} | updateQuery - {updateQuery}")
      self.abcDbObj.executeDmlQuery(updateQuery,params_update_batch_set)
      self.logger.info(f"{self.moduleName} | Updated batch set {batchSetKey} with status as {batchSetStatusInd}")
      
    except Exception:
      self.logger.exception(f"{self.moduleName} | Exception while updating batchset status for batchSetCategory:{batchSetCategory}, batchSetSrcSysNme:{batchSetSrcSysNme}")
      raise Exception 
  #------------------------------------------------------------------------------------------------------------------------------------#
  #======================================================GET PREVIOUS BATCH SET========================================================#
  #------------------------------------------------------------------------------------------------------------------------------------#  
  def getPreviousBatchset(self,batchSetCategory, batchSetSrcSysNme):
    try:  
      self.logger.debug(f"{self.moduleName} | -----getPreviousBatchset method START-------")
      self.logger.debug(f"{self.moduleName} | Getting the latest failed batch set  with batch_set_category - {batchSetCategory} and batch_set_src_sys_nme - {batchSetSrcSysNme}")

      #Retrieve details of previous run irrespective of status
      previousBatchSetQuery = f"SELECT * FROM  batch.t_batch_set WHERE batch_set_key = (SELECT max(batch_set_key) as batch_set_key FROM  batch.t_batch_set WHERE batch_set_key < (select max(batch_set_key) as batch_set_key from batch.t_batch_set  where batch_set_category = '{batchSetCategory}' and batch_set_src_sys_nme = '{batchSetSrcSysNme}')  AND batch_set_category = '{batchSetCategory}' and batch_set_src_sys_nme = '{batchSetSrcSysNme}') "
      self.logger.debug(f"{self.moduleName} | batch_set_fetch_query - {previousBatchSetQuery}")
      batchSetDetails = self.abcDbObj.executeSelectQuery(previousBatchSetQuery)
      self.logger.debug("-----getPreviousBatchset method END---------")
      return batchSetDetails
    except Exception:
      self.logger.exception(f"{self.moduleName} | Exception while getting previous batchset details for batchSetCategory:{batchSetCategory}, batchSetSrcSysNme:{batchSetSrcSysNme}")
      raise Exception       
      
  #------------------------------------------------------------------------------------------------------------------------------------#
  #====================================================OPEN BATCH======================================================================#
  #------------------------------------------------------------------------------------------------------------------------------------#    
  def openBatch(self, batchSetCategory, batchSetSrcSysNme, batchNme, batchSrcDbNme, batchTrgtDbNme, batchTrgtTblNme ):
    try:
      self.logger.info(f"{self.moduleName} | -----openBatch function START-------")
      self.logger.debug(f"{self.moduleName} | opening a batch with details -> batch_set_category - {batchSetCategory}, batch_set_src_sys_nme - {batchSetSrcSysNme} , batch_nme - {batchNme}, batch_src_db_nme - {batchSrcDbNme}, batch_trgt_db_nme - {batchTrgtDbNme}, batch_trgt_tbl_nme - {batchTrgtTblNme}")

      batchKey = None
      batchSetKey = None
      self.logger.debug(f"{self.moduleName} | fetching the active batch set details...")
      batchSetDetailsDf = self.getBatchSetInProgress(batchSetCategory, batchSetSrcSysNme)
      batchSetDetails = batchSetDetailsDf.collect()[0] if batchSetDetailsDf.count()>0 else None

      if batchSetDetails == None:
        raise Exception(f"{self.moduleName} | No active batch set available with batch_set_category - {batchSetCategory}, batch_set_src_sys_nme - {batchSetSrcSysNme} !!!")

      batchSetKey = batchSetDetails['batch_set_key']
      self.logger.info(f"{self.moduleName} | checking there is any batch in progress. Details => batch_set_category - {batchSetCategory}, batch_set_src_sys_nme - {batchSetSrcSysNme}, batch_nme - {batchNme}")
      batchInProgressDf = self.getBatchInProgress(batchSetCategory, batchSetSrcSysNme, batchNme)
      batchInProgress = batchInProgressDf.collect()[0] if batchInProgressDf.count()>0 else None

      if(batchInProgress != None):
        raise Exception(f"{self.moduleName} | There is a batch in progress... batch key - {batchInProgress['batch_key']} !!!")

      
      self.logger.info(f"{self.moduleName} | Creating new batch entry ...")
      params_open_batch = (batchSetKey,batchNme,batchSrcDbNme,batchTrgtDbNme,batchTrgtTblNme)
      insertQuery = f"""INSERT INTO [batch].[t_batch] ([batch_set_key], [batch_nme], [batch_strt_tm], [batch_end_tm], [batch_src_db_nme], [batch_trgt_db_nme], [batch_trgt_tbl_nme], [batch_trans_end], [batch_sts_ind], [batch_sts_descr]) VALUES (? ,? ,SYSDATETIME() ,NULL ,? ,? ,? ,NULL ,'P' ,'In progress' )"""
      self.logger.debug(f"batch insert query - {insertQuery}")
      self.abcDbObj.executeDmlQuery(insertQuery,params_open_batch)
      
      batchKey = self.getBatchInProgress(batchSetCategory, batchSetSrcSysNme, batchNme).collect()[0]['batch_key']
      self.logger.info(f"{self.moduleName} | Created new batch with batch key -> {batchKey} under batch set key -> {batchSetKey}")
      self.logger.info(f"{self.moduleName} | -----openBatch function END---------")
      return batchKey
    except Exception:
      self.logger.exception(f"{self.moduleName} | Exception while opening batch for batchSetCategory:{batchSetCategory}, batchSetSrcSysNme:{batchSetSrcSysNme}, batchNme:{batchNme}")
      raise Exception

  #------------------------------------------------------------------------------------------------------------------------------------#
  #=========================================================CLOSE BATCH================================================================#
  #------------------------------------------------------------------------------------------------------------------------------------#   
  # def closeBatch(self, batchSetCategory, batchSetSrcSysNme, batchNme, batchStatusCd, batchTransEnd=None, errorDetails = ""):
    
  #   try:
  #     self.logger.info(f"{self.moduleName} | -----closeBatch function START-------")
  #     batchStatusDict = {'S':'Success','F':'Failure'}
  #     if batchStatusCd in batchStatusDict.keys():
  #       batchStatusDesc = batchStatusDict[batchStatusCd]
  #     else:
  #       raise Exception(f"{self.moduleName} | Status given - {batchStatusCd}. This is invaild! .")

  #     batchInProgressDf = self.getBatchInProgress(batchSetCategory, batchSetSrcSysNme, batchNme)
  #     batchInProgress = batchInProgressDf.collect()[0] if batchInProgressDf.count()>0 else None

  #     if batchInProgress == None:
  #       raise Exception(f"{self.moduleName} | No batch in progress with details => batch_set_category - {batchSetCategory}, batch_set_src_sys_nme - {batchSetSrcSysNme}, batch_nme - {batchNme}.")

  #     batchKey = batchInProgress['batch_key']
  #     self.logger.info(f"{self.moduleName} | updating the status of batch entry(batch_key - {batchKey}) to status - {batchStatusCd}. batch details => batch_set_category - {batchSetCategory}, batch_set_src_sys_nme - {batchSetSrcSysNme}, batch_nme - {batchNme}.")
  #     #currentLoadIncrementalvalue=self.getCurrentIncrementalValue(batchKey)
  #     currentLoadIncrementalvalue="NULL" if batchTransEnd == None else f"'{str(batchTransEnd)}'"
  #     updateQuery =  f"UPDATE batch.t_batch SET batch_end_tm = CURRENT_TIMESTAMP,batch_trans_end = {currentLoadIncrementalvalue}, batch_sts_ind = '" + str(batchStatusCd) + "', batch_sts_descr = '" + str(batchStatusDesc) + "' WHERE batch_key = " + str(batchKey)
  #     # updateQuery = f"""UPDATE [batch].[t_batch] SET [batch_end_tm] = SYSDATETIME(),[batch_trans_end] = 1 ,[batch_sts_ind] =  '{batchStatusCd}'
  #     #     ,[batch_sts_descr] = '{batchStatusDesc}' WHERE batch_key = {batchKey}"""
  #     self.logger.debug(f"{self.moduleName} | update_query - {updateQuery}")
  #     self.abcDbObj.executeDmlQuery(updateQuery)
  #     self.logger.debug(f"{self.moduleName} | -----closeBatch function END-------")
  #   except Exception:
  #     self.logger.exception(f"{self.moduleName} | Exception while closing batch for batchSetCategory:{batchSetCategory}, batchSetSrcSysNme:{batchSetSrcSysNme}, batchNme:{batchNme}")
  #     raise Exception

  def closeBatch(self, batchSetCategory, batchSetSrcSysNme, batchNme, batchStatusCd, batchTransStart=None, errorDetails = ""):
    
    try:
      self.logger.info(f"{self.moduleName} | -----closeBatch function START-------")
      batchStatusDict = {'S':'Success','F':'Failure'}
      if batchStatusCd in batchStatusDict.keys():
        batchStatusDesc = batchStatusDict[batchStatusCd]
      else:
        raise Exception(f"{self.moduleName} | Status given - {batchStatusCd}. This is invaild! .")

      batchInProgressDf = self.getBatchInProgress(batchSetCategory, batchSetSrcSysNme, batchNme)
      batchInProgress = batchInProgressDf.collect()[0] if batchInProgressDf.count()>0 else None

      if batchInProgress == None:
        raise Exception(f"{self.moduleName} | No batch in progress with details => batch_set_category - {batchSetCategory}, batch_set_src_sys_nme - {batchSetSrcSysNme}, batch_nme - {batchNme}.")

      batchKey = batchInProgress['batch_key']
      self.logger.info(f"{self.moduleName} | updating the status of batch entry(batch_key - {batchKey}) to status - {batchStatusCd}. batch details => batch_set_category - {batchSetCategory}, batch_set_src_sys_nme - {batchSetSrcSysNme}, batch_nme - {batchNme}.")
      #currentLoadIncrementalvalue=self.getCurrentIncrementalValue(batchKey)
      currentLoadIncrementalvalue = "NULL" if batchTransStart is None else f"'{str(batchTransStart)}'"
      
      params_close_batch=(currentLoadIncrementalvalue,batchStatusCd,batchStatusDesc,batchKey)
      updateQuery = f"UPDATE batch.t_batch SET batch_end_tm = CURRENT_TIMESTAMP, batch_trans_strt = ?, batch_sts_ind = ?, batch_sts_descr = ? WHERE batch_key = ?"
      # updateQuery = f"""UPDATE [batch].[t_batch] SET [batch_end_tm] = SYSDATETIME(),[batch_trans_end] = 1 ,[batch_sts_ind] =  '{batchStatusCd}'
      #     ,[batch_sts_descr] = '{batchStatusDesc}' WHERE batch_key = {batchKey}"""
      
      self.logger.debug(f"{self.moduleName} | update_query - {updateQuery}")
      self.abcDbObj.executeDmlQuery(updateQuery,params_close_batch)
      self.logger.debug(f"{self.moduleName} | -----closeBatch function END-------")
    except Exception:
      self.logger.exception(f"{self.moduleName} | Exception while closing batch for batchSetCategory:{batchSetCategory}, batchSetSrcSysNme:{batchSetSrcSysNme}, batchNme:{batchNme}")
      raise Exception


  #------------------------------------------------------------------------------------------------------------------------------------#
  #====================================================GET FAILED BATCH DETAILS========================================================#
  #------------------------------------------------------------------------------------------------------------------------------------#  
  def getFailedBatchDetails(self,batchSetKey):
    try:
      self.logger.debug(f"{self.moduleName} | -----getFailedBatchDetails method START-------")
      self.logger.debug(f"{self.moduleName} | Getting the failed batch details for batch set -->{batchSetKey}")

      #Retrieve details of failed jobs in the previous run
      batchFetchQuery = f"SELECT * FROM  batch.t_batch WHERE batch_set_key={batchSetKey} AND batch_sts_ind='F' "
      self.logger.debug(f"{self.moduleName} | batch_set_fetch_query - {batchFetchQuery}")
      batchDetails = self.abcDbObj.executeSelectQuery(batchFetchQuery)
      self.logger.debug(f"{self.moduleName} | -----getFailedBatchDetails method END---------")
      return batchDetails  
    except Exception:
      self.logger.exception(f"{self.moduleName} | Exception while getting failed batch details details for batchSetKey:{batchSetKey} ")
      raise Exception

      
  #------------------------------------------------------------------------------------------------------------------------------------#
  #=================================================DATABRICKS INGESTION BALANCE MODULE================================================#
  #------------------------------------------------------------------------------------------------------------------------------------#
  def isBalanceMeasuresMatching(self,srcMeasuresDf,tgtMeasuresDf, batchSetCategory, batchSetSrcSysNme, batchNme):
    try:
      self.logger.info(f"{self.moduleName} | ----------------------isBalanceMeasuresMatching START-------------------------")
      #Final indicator of all balance checks , this will be set to False if any of the measure is not matching
      balanceMeasuresMatchingStatus = True
      

      srcMeasuresDfUpperCased = srcMeasuresDf.select([col.upper() for col in srcMeasuresDf.columns])
      tgtMeasuresDfUpperCased = tgtMeasuresDf.select([col.upper() for col in tgtMeasuresDf.columns])

      if srcMeasuresDfUpperCased.columns != tgtMeasuresDfUpperCased.columns:
        raise Exception("Source And Target Balance dataframes Have Different name/number/order of Measure columns. Please Check the measure details.")
      
      batchInProgressDf = self.getBatchInProgress(batchSetCategory, batchSetSrcSysNme, batchNme)
      batchInProgress = batchInProgressDf.collect()[0] if batchInProgressDf.count() > 0 else None
      
      if batchInProgress == None:
        raise Exception(f"{self.moduleName} | No batch in progress with details --> batch_set_category - {batchSetCategory}, batch_set_src_sys_nme - {batchSetSrcSysNme}, batch_nme - {batchNme}.") 
        
      batchKey = batchInProgress['batch_key']
      self.logger.debug(f"{self.moduleName} | Batch Key in progess for {batchNme} : {batchKey}")
      
      self.logger.debug(f"{self.moduleName} | Source and Target Balance dataframe has same number of columns with Matching Names")
      tgtMeasures = tgtMeasuresDf.collect()[0].asDict()
      
      srcMeasuresUpperCased = srcMeasuresDfUpperCased.collect()[0].asDict()
      tgtMeasuresUpperCased = tgtMeasuresDfUpperCased.collect()[0].asDict()
      self.logger.debug(f"{self.moduleName} | {srcMeasuresUpperCased} | {tgtMeasuresUpperCased}")
      for measure in tgtMeasures:
        #Compare the actual measure values
        measureUpperCased = measure.upper()
        if srcMeasuresUpperCased[measureUpperCased] == tgtMeasuresUpperCased[measureUpperCased]:
          self.logger.info(f"{self.moduleName} | The measure :{measure} is matching between source and target,  value :{srcMeasuresUpperCased[measureUpperCased]}")
          balancedInd = 'Y'            
        else:
          self.logger.info(f"{self.moduleName} | The measure :{measure} is not matching between source and target. source value :{srcMeasuresUpperCased[measureUpperCased]} ,target value :{tgtMeasuresUpperCased[measureUpperCased]}")
          balancedInd = 'N'
          balanceMeasuresMatchingStatus = False

        #Frame the Balance Insert Query   
        params_balance_insert=(batchKey,measure,srcMeasuresUpperCased[measureUpperCased],tgtMeasuresUpperCased[measureUpperCased],balancedInd)
        balanceInsertQuery = f"""INSERT INTO [audit].[t_balance] ([batch_key], [measure_type], [measure_src_value] ,[measure_trgt_value] ,[balanced_ind] ,[create_dt] ,[last_update_dt], [bal_config_id])  VALUES (?,?,?,?,?,SYSDATETIME(), SYSDATETIME(), NULL)"""

        #Run the Insert query
        self.logger.debug(f"{self.moduleName} | Executing Insert Query for Balance Entry query : {balanceInsertQuery}")
        self.abcDbObj.executeDmlQuery(balanceInsertQuery,params_balance_insert)
        self.logger.debug(f"{self.moduleName} | Balance Entry Inserted Successfully")
      self.logger.debug(f"{self.moduleName} | ----------------------isBalanceMeasuresMatching END-------------------------")
      return balanceMeasuresMatchingStatus 
    except Exception:
      self.logger.exception(f"{self.moduleName} | Exception while doing balance check for--> batchSetCategory:{batchSetCategory}, batchSetSrcSysNme:{batchSetSrcSysNme}")
      raise Exception
 
 # =========GET AND SET PREVIOUS BATCH TRANS START===========#
    
  def getAndSetPreviousBatchTransStartTimestamp(self, batchSetSrcSysCd, batchSetSrcSysName, batchName, batchkey):
        try:
            self.logger.info(f"{self.moduleName} | -----getAndsetPreviousBatchTransStart method START-------" )
            self.logger.info(f"{self.moduleName} | Getting the previous trans start /incremental criteria value for {batchkey}" )

            # Retrieve details of previous trans start / incremental value
            previousTransStartQuery = f"  SELECT batch_trans_strt FROM batch.t_batch WHERE batch_key=(select max(batch_key) from batch.t_batch where batch_nme='{batchName}' and batch_sts_ind='S' and batch_set_key=(select MAX(batch_set_key) from batch.t_batch_set where batch_set_src_sys_cd='{batchSetSrcSysCd}' AND batch_set_src_sys_nme='{batchSetSrcSysName}' and batch_set_sts_ind='S'))" 
            self.logger.debug(f"{self.moduleName} | previousTransStartQuery - {previousTransStartQuery}"  )
            previousTransStartDf = self.abcDbObj.executeSelectQuery(previousTransStartQuery)
            previousTransStart = (previousTransStartDf.collect()[0]["batch_trans_strt"] if previousTransStartDf.count() > 0 else "0")
            previousTransStartTmstpStr = previousTransStart if self.is_valid_date(previousTransStart) else datetime.now() - timedelta(days=365)
            self.logger.info(f"{self.moduleName} | ----- timestamp value fetched from previous trans start : {previousTransStartTmstpStr}-------" )

            #update value for batch_trans_strt in t_batch for the current batchKey
            #self.logger.info("-----UPDATE value for batch_trans_strt in batch.t_batch table---------")
            #updateQuery = f"UPDATE batch.t_batch SET batch_trans_strt = '"+str(previousTransEndTmstpStr)+"' WHERE batch_key = " + str(batchkey)
            #self.logger.debug(f"batch insert query - {updateQuery}")
            #self.abcDbObj.executeDmlQuery(updateQuery)

            self.logger.info("-----getPreviousBatchTransStart method END---------")
            return previousTransStartTmstpStr
        except Exception:
            self.logger.exception( f"{self.moduleName} | Exception while getting previous batch trans start details for batchKey:{batchkey}"  )
            raise Exception

 # =========GET AND SET PREVIOUS BATCH TRANS END===========#
    
  def getAndSetPreviousBatchTransEndTimestamp(self, batchSetSrcSysCd, batchSetSrcSysName, batchName, batchkey):
        try:
            self.logger.info(f"{self.moduleName} | -----getAndsetPreviousBatchTransEnd method START-------" )
            self.logger.info(f"{self.moduleName} | Getting the previous trans end /incremental criteria value for {batchkey}" )

            # Retrieve details of previous trans end / incremental value
            previousTransEndQuery = f"  SELECT batch_trans_end FROM batch.t_batch WHERE batch_key=(select max(batch_key) from batch.t_batch where batch_nme='{batchName}' and batch_sts_ind='S' and batch_set_key=(select MAX(batch_set_key) from batch.t_batch_set where batch_set_src_sys_cd='{batchSetSrcSysCd}' AND batch_set_src_sys_nme='{batchSetSrcSysName}' and batch_set_sts_ind='P'))" #batch_set_sts_ind='P' needs to be changed to 'S' ideally for batch.t_batch_set
            self.logger.debug(f"{self.moduleName} | previousTransEndQuery - {previousTransEndQuery}"  )
            previousTransEndDf = self.abcDbObj.executeSelectQuery(previousTransEndQuery)
            previousTransEnd = (previousTransEndDf.collect()[0]["batch_trans_end"] if previousTransEndDf.count() > 0 else "0")
            previousTransEndTmstpStr = previousTransEnd if self.is_valid_date(previousTransEnd) else "1900-01-01"
            self.logger.info(f"{self.moduleName} | ----- timestamp value fetched from previous trans end : {previousTransEndTmstpStr}-------" )

            #update value for batch_trans_strt in t_batch for the current batchKey
            self.logger.info("-----UPDATE value for batch_trans_strt in batch.t_batch table---------")
            param_update_previousTransEnd = (str(previousTransEndTmstpStr),batchkey)
            updateQuery = f"UPDATE batch.t_batch SET batch_trans_strt = ? WHERE batch_key = ?"
            self.logger.debug(f"batch insert query - {updateQuery}")
            self.abcDbObj.executeDmlQuery(updateQuery,param_update_previousTransEnd)

            self.logger.info("-----getPreviousBatchTransEnd method END---------")
            return previousTransEndTmstpStr
        except Exception:
            self.logger.exception( f"{self.moduleName} | Exception while getting previous batch trans end details for batchKey:{batchkey}"  )
            raise Exception

# =========IS VALID DATE CHECK===========#      
  def is_valid_date(self, date_string):
      if date_string is None:
        return False
      try:
          datetime.strptime(date_string, "%Y-%m-%d %H:%M:%S.%f")
          return True
      except ValueError:
          return False

# ======GET CURRENT LOAD INCREMENTAL VALUE===========#
    
  def getCurrentIncrementalValue(self, batchKey):
        try:
            self.logger.info(f"{self.moduleName} | -----getCurrentIncrementalValue method START-------" )
            self.logger.debug(f"{self.moduleName} | Getting the current load incremental value for batch_key - {batchKey}" ) 
            incrementalQuery = f"select batch_trans_strt from batch.t_batch where batch_key = {batchKey}"
            self.logger.debug( f"{self.moduleName} | current load incremental query - {incrementalQuery}" )
            currentIncrementalvalue = self.abcDbObj.executeSelectQuery(incrementalQuery).collect()[0][0]
            self.logger.info("-----getCurrentIncrementalValue method END---------")
            return currentIncrementalvalue
        except Exception:
            self.logger.exception(
                f"{self.moduleName} | Exception while retreiving in getCurrentIncrementalValue for batch_key - {batchKey}. "
            )
            raise Exception
  def getMaxSuccessBatchset(self, batchSetCategory, batchSetSrcSysNme):
      try:
          self.logger.info(f"{self.moduleName} | -----getMaxSuccessBatchset method START-------" )
          self.logger.info(f"{self.moduleName} | Getting the latest failed batch set  with batch_set_category - {batchSetCategory} and batch_set_src_sys_nme - {batchSetSrcSysNme}" )

          # Retrieve details of Maximum Success run 
          MaxSuccessBatchSetQuery = f"SELECT * FROM  batch.t_batch_set WHERE batch_set_key = (SELECT max(batch_set_key) as batch_set_key FROM  batch.t_batch_set WHERE batch_set_sts_ind='S'  AND batch_set_category = '{batchSetCategory}' and batch_set_src_sys_nme = '{batchSetSrcSysNme}') "
          self.logger.debug(f"{self.moduleName} | batch_set_fetch_query - {MaxSuccessBatchSetQuery}"  )
          batchSetDetails = self.abcDbObj.executeSelectQuery(MaxSuccessBatchSetQuery)
          self.logger.info("-----getMaxSuccessBatchset method END---------")
          return batchSetDetails
      except Exception:
          self.logger.exception( f"{self.moduleName} | Exception while getting MaxSuccess batchset details for batchSetCategory:{batchSetCategory}, batchSetSrcSysNme:{batchSetSrcSysNme}"  )
          raise Exception
# =========OPEN BATCH RECON ======================
  def openBatchRecon(self,batchSetCategory,batchSetSrcSysNme, batchSetSrcSysCd,batchNme,batchSrcDbNme,batchTrgtDbNme,batchTrgtTblNme,FactInd="N"):
        try:
            self.logger.info(f"{self.moduleName} | -----openBatchRecon function START-------")
            self.logger.debug( f"{self.moduleName} | opening a batch with details -> batch_set_category - {batchSetCategory}, batch_set_src_sys_nme - {batchSetSrcSysNme} , batch_nme - {batchNme}, batch_src_db_nme - {batchSrcDbNme}, batch_trgt_db_nme - {batchTrgtDbNme}, batch_trgt_tbl_nme - {batchTrgtTblNme}")

            batchKey = None
            batchSetKey = None
            self.logger.debug(f"{self.moduleName} | fetching the active batch set details...")
            batchSetDetailsDf = self.getMaxSuccessBatchset(batchSetCategory, batchSetSrcSysNme) 
            batchSetDetails = batchSetDetailsDf.collect()[0] if batchSetDetailsDf.count() > 0 else None 
            
            if batchSetDetails == None:
                raise Exception(f"{self.moduleName} | No maximum success batch set available with batch_set_category - {batchSetCategory}, batch_set_src_sys_nme - {batchSetSrcSysNme} !!!"  )

            batchSetKey = batchSetDetails["batch_set_key"]
            self.logger.debug(f"{self.moduleName} | checking there is any batch in progress. Details => batch_set_category - {batchSetCategory}, batch_set_src_sys_nme - {batchSetSrcSysNme}, batch_nme - {batchNme}" )
            batchInProgressDf = self.getBatchInProgress( batchSetCategory, batchSetSrcSysNme, batchNme )
            batchInProgress = ( batchInProgressDf.collect()[0] if batchInProgressDf.count() > 0 else None )
            
            if batchInProgress != None:
                raise Exception(f"{self.moduleName} | There is a batch in progress... batch key - {batchInProgress['batch_key']} !!!" )

            self.logger.debug(f"{self.moduleName} | Creating new batch entry ...")

            #f-string cant handle null for transactionStart variable, and hence concatenation method is used
            param_open_batch_recon=(batchSetKey,batchNme,batchSrcDbNme,batchTrgtDbNme,batchTrgtTblNme)
            insertQuery = "INSERT INTO batch.t_batch (batch_set_key, batch_nme, batch_strt_tm, batch_end_tm, batch_src_db_nme, batch_trgt_db_nme, batch_trgt_tbl_nme,batch_trans_strt, batch_trans_end, batch_sts_ind, batch_sts_descr) VALUES (?,?,CURRENT_TIMESTAMP ,null ,?,?,?,null,null ,'P','In progress')"

            self.logger.debug(f"batch insert query - {insertQuery}")
            self.abcDbObj.executeDmlQuery(insertQuery,param_open_batch_recon)

            batchKey = self.getBatchInProgress( batchSetCategory, batchSetSrcSysNme, batchNme).collect()[0]["batch_key"]
            self.logger.info(f"{self.moduleName} | Created new batch with batch key -> {batchKey} under batch set key -> {batchSetKey}")
            self.logger.info( f"{self.moduleName} | -----openBatchRecon function END---------" )
            return batchKey
        except Exception:
            self.logger.exception( f"{self.moduleName} | Exception while opening batch for batchSetCategory:{batchSetCategory}, batchSetSrcSysNme:{batchSetSrcSysNme}, batchNme:{batchNme}")
            raise Exception
          
  def closeBatchRecon(self, batchSetCategory, batchSetSrcSysNme, batchNme, batchStatusCd,batchTransEnd=None,errorDetails=None):
        try:
            self.logger.info( f"{self.moduleName} | -----closeBatchRecon function START-------"      )
            batchStatusDict = {"S": "Success", "F": "Failure"}
            if batchStatusCd in batchStatusDict.keys():
                batchStatusDesc = batchStatusDict[batchStatusCd]
            else:
                raise Exception(  f"{self.moduleName} | Status given - {batchStatusCd}. This is invaild! ."           )

            batchInProgressDf = self.getBatchInProgress(  batchSetCategory, batchSetSrcSysNme, batchNme        )
            batchInProgress = ( batchInProgressDf.collect()[0] if batchInProgressDf.count() > 0    else None        )

            if batchInProgress == None:
                raise Exception(  f"{self.moduleName} | No batch in progress with details => batch_set_category - {batchSetCategory}, batch_set_src_sys_nme - {batchSetSrcSysNme}, batch_nme - {batchNme}."      )

            batchKey = batchInProgress["batch_key"]
            self.logger.info(      f"{self.moduleName} | updating the status of batch entry(batch_key - {batchKey}) to status - {batchStatusCd}. batch details => batch_set_category - {batchSetCategory}, batch_set_src_sys_nme - {batchSetSrcSysNme}, batch_nme - {batchNme}."         )

            #f-string cant handle null, and hence concatenation option is used
            batchTransEndStr="null" if batchTransEnd == None else str(batchTransEnd)
            param_close_batch_recon=(batchTransEndStr,batchStatusCd,batchStatusDesc,batchKey)
            updateQuery = "UPDATE batch.t_batch SET batch_end_tm = CURRENT_TIMESTAMP,batch_trans_end = ?, batch_sts_ind =?, batch_sts_descr = ? WHERE batch_key = ?"
            
            self.logger.debug(f"{self.moduleName} | update_query - {updateQuery}")
            self.abcDbObj.executeDmlQuery(updateQuery,param_close_batch_recon)
            self.logger.info(f"{self.moduleName} | -----closeBatchRecon function END-------"       )
        except Exception:
            self.logger.exception(
                f"{self.moduleName} | Exception while closing batch for batchSetCategory:{batchSetCategory}, batchSetSrcSysNme:{batchSetSrcSysNme}, batchNme:{batchNme}"
            )
            raise Exception